# 06 · eval — **외부 baseline** (+ `act_te`)

**150k 체크포인트 × 5회 반복.** rep 마다 `--seed = 1000 + 100·rep` → env 초기상태가 달라짐.
- best-ckpt 를 안 고르는 이유: 모델마다 유리한 step 을 뽑으면 cherry-pick → 전 모델 동일 step(150k).
- rep 마다 seed 를 바꾸는 이유: 같은 seed 로 5번 돌리면 결정적이라 반복이 무의미.
  학습 seed(모델 분산)와 rep(평가 분산)이 분리됨.

`act_te` = **act 체크포인트 재사용** + eval-time temporal ensembling(coeff 0.01, n_action_steps 1).
학습 안 함. ACT 자신의 smoothing 이라 **매끄러움 비교의 기준선**.

5모델 × 4 seed × 5 rep = **100 run**.

끝난 run 은 자동 skip → 중단/재실행 안전.


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

TASK  = cf.MAIN_SIM        # 'insertion' (aloha)
SEEDS = cf.MAIN_SEEDS      # [0,1,2,3] — 은지와 분담하면 여기만 바꿈 (예: [0,1])
GPUS  = cf.v23.available_gpus()   # 이 노드에 실제 보이는 GPU
NGPU  = len(GPUS)                 # 4개면 4잡씩 청크로 (하드코딩 X)
TAGS = cf.GROUP_BASELINE + ['act_te']   # act_te 는 act ckpt 를 재사용
REPS = list(range(cf.EVAL_REPEATS))   # [0..4]
N_EP = cf.EVAL_N_EP                   # rep 1회당 에피소드 (부담되면 30)

print('GPU :', GPUS, f'({NGPU}개)')
print('eval:', TAGS, '| ckpt', f'{cf.CKPT_STEP:,}', '| reps', REPS, '| n_ep', N_EP)
print('총 run:', len(TAGS) * len(SEEDS) * len(REPS))

## 사전 확인 — 150k 체크포인트

In [ ]:
ok = cf.print_ckpt_status(cf.GROUP_BASELINE, SEEDS, TASK)

## 반복 eval 실행

In [ ]:
cf.run_repeat_evals(TAGS, SEEDS, REPS, task=TASK, ngpu=NGPU, n_episodes=N_EP)

## 결과 (SR) — 전체 표/그림은 `09_report_sr`, 떨림은 `10_report_jerk`

In [ ]:
rows = cf.sr_table(TAGS, SEEDS, REPS, task=TASK, n_episodes=N_EP)